# Taller: Deep Learning con Series de Tiempo (EEG Motor Imagery) - LSTM

## 1. Introducción

Motor imagery (imaginación motora) es una técnica ampliamente utilizada en neuroingeniería y en interfaces cerebro–computador (BCI). Consiste en imaginar la ejecución de un movimiento sin realizarlo físicamente, lo que activa patrones eléctricos cerebrales similares a los generados durante el movimiento real.

Durante la imaginación de mover una mano, por ejemplo, se produce una modulación característica en las áreas motoras del cerebro (principalmente en la corteza motora contralateral), observable en el EEG como cambios en los ritmos mu (8–13 Hz) y beta (13–30 Hz). Estas variaciones se conocen como event-related desynchronization/synchronization (ERD/ERS).

El objetivo en los experimentos de MI es distinguir automáticamente entre distintas clases mentales (por ejemplo, imaginar mover la mano izquierda vs. la derecha) a partir de estas señales EEG. Para ello, se aplican modelos de aprendizaje automático y, más recientemente, redes neuronales profundas que pueden aprender directamente patrones espacio–temporales.

En este taller trabajaremos con un subconjunto de datos de EEG registrados en los electrodos C3, Cz y C4, situados sobre el área motora, para entrenar un modelo LSTM (Long Short-Term Memory) capaz de identificar el tipo de imaginación motora representada en cada ensayo

## 2. Objetivos
Objetivo general aplicar un flujo completo de DL sobre series de tiempo:
1. Visualización de señales
2. Verificación de balance
3. Creación de Dataset
4. Entrenamiento de un LSTM
5. Análisis de sobreajuste variando hiperparámetros.


## 3. Dataset


El conjunto de datos [BCI Competition IV](https://www.bbci.de/competition/iv/) – Dataset 2b fue desarrollado por el Laboratory of Brain-Computer Interfaces en Graz University of Technology (Austria). Este dataset fue diseñado para evaluar algoritmos de clasificación de motor imagery (MI) en condiciones realistas y mínimamente instrumentadas, simulando un sistema BCI práctico. Contiene registros de EEG cued motor imagery, es decir, tareas de imaginación motora guiadas por una señal visual en las que los participantes imaginaban mover su mano izquierda o derecha siguiendo las instrucciones presentadas en pantalla.

Las características principales del conjunto son:

* Sujetos: 9 voluntarios sanos.
* Canales EEG: 3 bipolares (C3, Cz y C4) ubicados sobre la corteza motora.
* Canales EOG: 3 adicionales para registrar artefactos oculares.
* Frecuencia de muestreo: 250 Hz.
* Filtros aplicados: 0.5–100 Hz (con notch a 50 Hz).
* Clases: dos categorías (imaginación de movimiento de la mano izquierda (0) y mano derecha (1)).

Para este taller, se entregará una matriz `X` con las señales EEG preprocesadas y una matriz `Y` con las etiquetas correspondientes (0 o 1). Cada muestra de `X` representa un ensayo individual de imaginación motora, correspondiente a un intervalo de 1 segundo de señal EEG. En este intervalo, el sujeto se encontraba realizando la tarea de imaginar el movimiento indicado (izquierda o derecha).

Por lo tanto:

* `X` tiene forma `(N, C, T)`, donde

  * `N` es el número total de ensayos,
  * `C` corresponde a los tres canales EEG (C3, Cz, C4), y
  * `T` es el número de puntos temporales en cada ensayo (750 muestras en 3 segundos). 
* `Y` contiene las etiquetas binarias asociadas a cada ensayo (`0` = mano izquierda, `1` = mano derecha).

A partir de estos datos, podremos visualizar las señales crudas, verificar el balance de clases, entrenar modelos de deep learning y evaluar su desempeño.


![](img/eeg.png)




In [ ]:

%%
!wget -q https://www.bbci.de/competition/download/competition_iv/BCICIV_2b_gdf.zip
!unzip -q BCICIV_2b_gdf.zip
import mne, os, numpy as np

X, Y = [], []
archivos = [f for f in os.listdir() if f.endswith(".gdf")]

for archivo in archivos:
    raw = mne.io.read_raw_gdf(archivo, verbose=False, include=['EEG:C3','EEG:C4','EEG:Cz'])
    sf = raw.info['sfreq']
    for annot in raw.annotations:
        if annot['description'] in ['769','770']:
            onset = int(annot['onset'] * sf)
            seg = raw.get_data(start=onset, stop=onset + int(1*sf), units='uV')  # (C,T)
            X.append(seg)
            Y.append(0 if annot['description']=='769' else 1)

X = np.array(X)  # (N,C,T)
Y = np.array(Y)  # (N,)
# Normalización global (z-score)
X = (X - X.mean()) / (X.std() + 1e-8)
print("Shapes:", X.shape, Y.shape, "Mean:", X.mean().round(4), "Std:", X.std().round(4))


# %% [markdown]
# ## 1. Visualización de la data
# **Ejercicio 1**
# - Grafica 3 ensayos de la **clase 0** y 3 de la **clase 1** (los tres canales).
# - Haz un `subplot` por canal (C3, Cz, C4). Añade leyenda y títulos.

# %%
# <CODE>
import matplotlib.pyplot as plt

canales = ["C3","Cz","C4"]
fig, axes = plt.subplots(3, 1, figsize=(10,6), sharex=True)
for ch in range(3):
    idx0 = np.where(Y==0)[0][:3]
    idx1 = np.where(Y==1)[0][:3]
    for i in idx0:
        axes[ch].plot(X[i, ch], alpha=0.7, label="clase 0" if i==idx0[0] else None)
    for i in idx1:
        axes[ch].plot(X[i, ch], alpha=0.7, linestyle="--", label="clase 1" if i==idx1[0] else None)
    axes[ch].set_title(f"Canal {canales[ch]}")
    axes[ch].grid(True)
axes[-1].set_xlabel("Muestras (t)")
axes[0].legend(loc="upper right")
plt.tight_layout(); plt.show()


# %% [markdown]
# ## 2. Verificación del balance de clases
# **Ejercicio 2**
# - Cuenta instancias por clase y grafícalas en un **bar plot**.
# - Comenta si hay desbalance y cómo podría afectar el entrenamiento.

# %%
# <CODE>
vals, cnts = np.unique(Y, return_counts=True)
print("Conteo por clase:", dict(zip(vals, cnts)))
plt.figure(figsize=(4,3))
plt.bar([str(v) for v in vals], cnts)
plt.title("Distribución de clases")
plt.xlabel("Clase"); plt.ylabel("N")
plt.grid(axis='y'); plt.show()


# %% [markdown]
# ## 3. Dataset y DataLoader
# **Ejercicio 3**
# - Implementa un `Dataset` con `__getitem__` que entregue `(x,y)` como tensores.
# - Separa **train/test** (80/20) estratificado.
# - Crea `DataLoader` con `batch_size=32`. `shuffle=True` sólo en train.

# %%
# <CODE>
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

class EEGDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)   # (N,C,T)
        self.Y = torch.tensor(Y, dtype=torch.float32)   # (N,)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

X_tr, X_te, y_tr, y_te = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

train_ds = EEGDataset(X_tr, y_tr)
test_ds  = EEGDataset(X_te, y_te)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=torch.cuda.is_available())
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())

print(f"Train: {len(train_ds)} | Test: {len(test_ds)}")


# %% [markdown]
# ## 4. LSTM para clasificación binaria
# **Ejercicio 4**
# - Implementa un LSTM que reciba secuencias `(B, T, C)` y devuelva logits `(B,1)`.
# - Entrénalo **50 épocas** con `Adam(lr=1e-3)` y `BCEWithLogitsLoss`.
# - Registra `loss` y `accuracy` en train/test y grafícalos.

# %%
# <CODE>
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

class RNNBiomed(nn.Module):
    def __init__(self, input_size=3, hidden_size=32, num_layers=1, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout if num_layers>1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)  # binario

    def forward(self, x):
        # x: (B, C, T) -> (B, T, C)
        x = x.transpose(1, 2)
        out, _ = self.lstm(x)     # (B, T, H)
        out = out[:, -1, :]       # último paso temporal
        out = self.dropout(out)
        return self.fc(out)       # (B,1) logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = RNNBiomed(input_size=3, hidden_size=32, num_layers=1, dropout=0.3).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

num_epochs = 50
train_losses, test_losses = [], []
train_accs, test_accs = [], []

for epoch in range(num_epochs):
    # --- train ---
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device).unsqueeze(1)  # (B,1)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)  # opcional
        optimizer.step()

        running_loss += loss.item()*xb.size(0)
        preds = (torch.sigmoid(logits) > 0.5).float()
        correct += (preds == yb).sum().item()
        total += yb.size(0)

    tr_loss = running_loss / total
    tr_acc  = correct / total * 100

    # --- eval ---
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device).unsqueeze(1)
            logits = model(xb)
            loss = criterion(logits, yb)
            running_loss += loss.item()*xb.size(0)
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == yb).sum().item()
            total += yb.size(0)

    te_loss = running_loss / total
    te_acc  = correct / total * 100

    train_losses.append(tr_loss); test_losses.append(te_loss)
    train_accs.append(tr_acc);    test_accs.append(te_acc)
    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {tr_loss:.4f} | Train Acc: {tr_acc:.2f}% "
          f"- Test Loss: {te_loss:.4f} | Test Acc: {te_acc:.2f}%")

# Curvas
fig, ax = plt.subplots(1,2, figsize=(12,5))
ax[0].plot(train_losses, label="Train Loss"); ax[0].plot(test_losses, label="Test Loss")
ax[0].set_title("Loss"); ax[0].set_xlabel("Epoch"); ax[0].legend(); ax[0].grid(True)
ax[1].plot(train_accs, label="Train Acc"); ax[1].plot(test_accs, label="Test Acc")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("Epoch"); ax[1].legend(); ax[1].grid(True)
plt.tight_layout(); plt.show()


# %% [markdown]
# ## 5. Diagnóstico de overfitting
# **Ejercicio 5**
# - Modifica y prueba las siguientes configuraciones (una a la vez) y comenta efectos en las curvas:
#   - `hidden_size`: 16, 32, 64
#   - `num_layers`: 1, 2
#   - `dropout`: 0.0, 0.3, 0.5
#   - `weight_decay` (Adam): 0, 1e-5, 1e-4
#   - `max_norm` en `clip_grad_norm_`: 1.0, 5.0 (o quítalo)
# - Señales de **overfit**: `Train Acc ↑` mientras `Test Acc ↔/↓`, y `Train Loss ↓` más que `Test Loss`.
# - ¿Qué combinación reduce la brecha Train–Test?

# %%
# <CODE> Plantilla rápida para reintentos
def train_eval(hidden_size=32, num_layers=1, dropout=0.3, weight_decay=1e-5, epochs=20):
    model = RNNBiomed(input_size=3, hidden_size=hidden_size, num_layers=num_layers, dropout=dropout).to(device)
    opt = optim.Adam(model.parameters(), lr=1e-3, weight_decay=weight_decay)
    trL, teL, trA, teA = [], [], [], []
    for ep in range(epochs):
        # train
        model.train(); runL, cor, tot = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device).unsqueeze(1)
            opt.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            opt.step()
            runL += loss.item()*xb.size(0)
            preds = (torch.sigmoid(logits)>0.5).float()
            cor += (preds==yb).sum().item(); tot += yb.size(0)
        trL.append(runL/tot); trA.append(cor/tot*100)
        # test
        model.eval(); runL, cor, tot = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(device), yb.to(device).unsqueeze(1)
                logits = model(xb)
                loss = criterion(logits, yb)
                runL += loss.item()*xb.size(0)
                preds = (torch.sigmoid(logits)>0.5).float()
                cor += (preds==yb).sum().item(); tot += yb.size(0)
        teL.append(runL/tot); teA.append(cor/tot*100)
    return trL, teL, trA, teA

# Ejemplo rápido:
# trL, teL, trA, teA = train_eval(hidden_size=64, num_layers=2, dropout=0.5, weight_decay=1e-4, epochs=15)
# plt.figure(figsize=(10,4)); plt.subplot(1,2,1); plt.plot(trL); plt.plot(teL); plt.title("Loss");
# plt.subplot(1,2,2); plt.plot(trA); plt.plot(teA); plt.title("Acc"); plt.show()


# %% [markdown]
# ## 6. (Opcional) Buenas prácticas
# - **Early Stopping**: detener cuando `val_loss` no mejora n épocas.
# - **Batch/Layer Norm**: estabilizar activaciones.
# - **Aumento de datos temporal**: ruido leve, jitter, recorte temporal.
# - **Balance de clases**: `WeightedRandomSampler` o `pos_weight` en `BCEWithLogitsLoss`.
#
# Con esto ya tienes la receta completa para DL en series de tiempo EEG con LSTM.
